# Mục 3b(ii) — Chọn thuộc tính làm mịn
**Võ Minh Huy — 2311110117**

Đối chiếu bước SQL chọn thuộc tính có miền giá trị lớn nhất và độ lệch chuẩn lớn nhất. Dùng bản làm việc sau N07 do notebook 11 đã lưu; chưa thay đổi giá trị hoặc thực hiện làm mịn.

## 1. Thống kê các thuộc tính
Miền giá trị = Max − Min. Độ lệch chuẩn mẫu dùng `std(ddof=1)`, tương ứng `STDEV` của SQL. Khảo sát đúng 12 thuộc tính như SQL; loại mã hồ sơ, nhóm tuổi phát sinh và nhãn kết quả. Tính trên giá trị chưa làm tròn; chỉ làm tròn bảng hiển thị đến 2 chữ số.

In [1]:
from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP
import pandas as pd

thu_muc = Path.cwd()
if thu_muc.name == "python":
    thu_muc = thu_muc.parent
file_sau_n07 = thu_muc / "ket_qua" / "11_dien_ejection_fraction" / "ban_lam_viec_sau_ef_python.csv"
if not file_sau_n07.exists():
    raise FileNotFoundError("Chạy Run All notebook 11_dien_null_ejection_fraction.ipynb trước.")
du_lieu = pd.read_csv(file_sau_n07)
cac_cot = ["age", "anaemia", "creatinine_phosphokinase", "diabetes",
    "ejection_fraction", "high_blood_pressure", "platelets", "serum_creatinine",
    "serum_sodium", "sex", "smoking", "time"]

thong_ke = pd.DataFrame({"TenCot":cac_cot,
    "SoGiaTri":[du_lieu[c].count() for c in cac_cot],
    "Min":[du_lieu[c].min() for c in cac_cot],
    "Max":[du_lieu[c].max() for c in cac_cot],
    "MienGiaTri":[du_lieu[c].max()-du_lieu[c].min() for c in cac_cot],
    "DoLechChuan":[du_lieu[c].std(ddof=1) for c in cac_cot]
}).sort_values(["MienGiaTri", "TenCot"], ascending=[False, True], ignore_index=True)

def hien_thi_2_so(value):
    return Decimal(str(value)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)

thong_ke_hien_thi = thong_ke.copy()
for c in ["Min", "Max", "MienGiaTri", "DoLechChuan"]:
    thong_ke_hien_thi[c] = thong_ke_hien_thi[c].map(hien_thi_2_so)
thong_ke_hien_thi

,TenCot,SoGiaTri,Min,Max,MienGiaTri,DoLechChuan
0,platelets,299,25100.00,850000.00,824900.00,94807.77
1,creatinine_phosphokinase,299,23.00,7861.00,7838.00,958.09
2,time,299,4.00,285.00,281.00,77.61
3,ejection_fraction,299,14.00,80.00,66.00,11.83
4,age,299,40.00,95.00,55.00,11.25
5,serum_sodium,299,113.00,148.00,35.00,4.30
6,serum_creatinine,299,0.50,9.40,8.90,1.03
7,anaemia,299,0.00,1.00,1.00,0.50
8,diabetes,299,0.00,1.00,1.00,0.49
9,high_blood_pressure,299,0.00,1.00,1.00,0.48


## 2. Chọn theo hai tiêu chí riêng
Chọn từ thống kê chưa làm tròn. Nếu đồng hạng, sắp tên cột để chọn nhất quán với SQL. Bảng này chỉ xác định thuộc tính cho hai yêu cầu của đề; các phép làm mịn thực hiện ở bước sau.

In [2]:
theo_mien = thong_ke.sort_values(["MienGiaTri", "TenCot"], ascending=[False, True]).iloc[0]
theo_std = thong_ke.sort_values(["DoLechChuan", "TenCot"], ascending=[False, True]).iloc[0]
lua_chon = pd.DataFrame({
    "TieuChi":["Miền giá trị lớn nhất", "Độ lệch chuẩn lớn nhất"],
    "ThuocTinh":[theo_mien["TenCot"], theo_std["TenCot"]],
    "GiaTriTieuChi":[hien_thi_2_so(theo_mien["MienGiaTri"]),
                    hien_thi_2_so(theo_std["DoLechChuan"])]
})
lua_chon

,TieuChi,ThuocTinh,GiaTriTieuChi
0,Miền giá trị lớn nhất,platelets,824900.00
1,Độ lệch chuẩn lớn nhất,platelets,94807.77


**Nhận xét:** platelets đứng đầu cả hai tiêu chí: Min = 25.100, Max = 850.000, miền giá trị = 824.900; độ lệch chuẩn mẫu ≈ 94.807,77. Chọn platelets cho bước chia ít nhất 5 nhóm và bước tăng/giảm 10% theo đề. Các phép biến đổi chưa được thực hiện trong notebook này.

## 3. Lưu thống kê

In [3]:
thu_muc_kq = thu_muc / "ket_qua" / "13_lam_min" / "01_chon_thuoc_tinh"
thu_muc_kq.mkdir(parents=True, exist_ok=True)
thong_ke_hien_thi.to_csv(thu_muc_kq / "thuoc_tinh_python.csv", index=False, encoding="utf-8-sig")
lua_chon.to_csv(thu_muc_kq / "lua_chon_python.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"TrangThai":["Đã lưu bảng khảo sát; chưa thay đổi dữ liệu sau N07."]})

,TrangThai
0,Đã lưu bảng khảo sát; chưa thay đổi dữ liệu sa...
